# Ridge Regression

- It is Linear Regression with an L2 regularization penalty on the coefficients.

```text
Linear Regression
      ↓
Minimize prediction error
```
---
```text
Ridge
      ↓
Minimize prediction error
+
penalize large coefficients
```

In [1]:
# Create a datset with correlated features.
import numpy as np
import pandas as pd

np.random.seed(42)

n = 100

x1 = np.random.randn(n)

# Highly correlated with x1
x2 = x1 + np.random.normal(0, 0.1, n)

x3 = np.random.randn(n)

y = 5 * x1 + 2 * x3 + np.random.normal(0, 1, n)

data = pd.DataFrame({
    "x1": x1,
    "x2": x2,
    "x3": x3,
    "y": y
})    

In [2]:
# inspect the first few rows of the dataset
data.head()

,x1,x2,x3,y
0,0.496714,0.355177,0.357787,2.370150
1,-0.138264,-0.180329,0.560785,-0.129933
2,0.647689,0.613417,1.083051,6.151839
3,1.523030,1.442802,1.053802,10.333124
4,-0.234153,-0.250282,-1.377669,-3.947007


In [3]:
# First we'll try to fit a linear regression model without any regularization.
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

X = data[["x1", "x2", "x3"]]
y = data["y"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

linear_model = LinearRegression()

linear_model.fit(X_train, y_train)

print("Coefficients:", linear_model.coef_)
print("Intercept:", linear_model.intercept_)

Coefficients: [ 5.504309   -0.71613748  2.0135245 ]
Intercept: 0.1534518481317787


In [ ]:
# Now we'll implement Ridge Regression to see how it handles the multicollinearity in the dataset.
from sklearn.linear_model import Ridge

ridge_model = Ridge(alpha=1.0) # alpha is the hyperparameter that controls the strength of the regularization. A higher alpha means more regularization.

ridge_model.fit(X_train, y_train)

print("Coefficients:", ridge_model.coef_)
print("Intercept:", ridge_model.intercept_)

Coefficients: [3.21669646 1.56040982 1.99512116]
Intercept: 0.1465958374858934


In [6]:
# Compare 
print("Linear:", linear_model.coef_)
print("Ridge :", ridge_model.coef_)

Linear: [ 5.504309   -0.71613748  2.0135245 ]
Ridge : [3.21669646 1.56040982 1.99512116]


In [7]:
# Experiment with different values of alpha to see how it affects the coefficients and the model's performance.
from sklearn.linear_model import Ridge

for alpha in [0, 0.01, 0.1, 1, 10, 100]:

    model = Ridge(alpha=alpha)

    model.fit(X_train, y_train)

    print(
        f"Alpha={alpha}, "
        f"Coefficients={model.coef_}"
    )

Alpha=0, Coefficients=[ 5.504309   -0.71613748  2.0135245 ]
Alpha=0.01, Coefficients=[ 5.4204254  -0.63179771  2.01325958]
Alpha=0.1, Coefficients=[ 4.82977451 -0.03857161  2.01109102]
Alpha=1, Coefficients=[3.21669646 1.56040982 1.99512116]
Alpha=10, Coefficients=[2.37236481 2.13918437 1.86553802]
Alpha=100, Coefficients=[1.45272911 1.41872574 1.13941053]


In [8]:
# check with predictions 
from sklearn.metrics import mean_squared_error
import numpy as np

for alpha in [0.01, 0.1, 1, 10, 100]:

    model = Ridge(alpha=alpha)

    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_rmse = np.sqrt(
        mean_squared_error(y_train, train_pred)
    )

    test_rmse = np.sqrt(
        mean_squared_error(y_test, test_pred)
    )

    print(
        f"Alpha={alpha:>5} | "
        f"Train RMSE={train_rmse:.3f} | "
        f"Test RMSE={test_rmse:.3f}"
    )

Alpha= 0.01 | Train RMSE=0.904 | Test RMSE=0.716
Alpha=  0.1 | Train RMSE=0.906 | Test RMSE=0.708
Alpha=    1 | Train RMSE=0.930 | Test RMSE=0.715
Alpha=   10 | Train RMSE=1.008 | Test RMSE=0.858
Alpha=  100 | Train RMSE=2.373 | Test RMSE=2.280


```text
alpha too small
      ↓
not enough regularization
      ↓
possible overfitting
```
---
```text
alpha appropriate
      ↓
good generalization
```
---
```text
alpha too large
      ↓
too much regularization
      ↓
underfitting
```

In [9]:
# Let's try polynomial + Ridge regression to see how it performs on a non-linear dataset.
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge

model = make_pipeline(PolynomialFeatures(degree=2), Ridge(alpha=1.0))

model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('polynomialfeatures', ...), ('ridge', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](3,)","['x1','x2','x3']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,3
,"degree degree: int or tuple (min_degree, max_degree), default=2If a single int is given, it specifies the maximal degree of thepolynomial features. If a tuple `(min_degree, max_degree)` is passed,then `min_degree` is the minimum and `max_degree` is the maximumpolynomial degree of the generated features. Note that `min_degree=0`and `min_degree=1` are equivalent as outputting the degree zero term isdetermined by `include_bias`.",2
,"interaction_only interaction_only: bool, default=FalseIf `True`, only interaction features are produced: features that areproducts of at most `degree` *distinct* input features, i.e. terms withpower of 2 or higher of the same input feature are excluded:- included: `x[0]`, `x[1]`, `x[0] * x[1]`, etc.- excluded: `x[0] ** 2`, `x[0] ** 2 * x[1]`, etc.",False
,"include_bias include_bias: bool, default=TrueIf `True` (default), then include a bias column, the feature in whichall polynomial powers are zero (i.e. a column of ones - acts as anintercept term in a linear model).",True
,"order order: {'C', 'F'}, default='C'Order of output array in the dense case. `'F'` order is faster tocompute, but may slow down subsequent estimators... versionadded:: 0.21",'C'


**Why do we scale Before Ridge?**
```text
Let's say we have 
age = 20-60
Salary = 20,000 to 2,00,000

we know that Ridge penalizes the coefficients.

So, if features are widly different scales, coefficient magnitude isn't directly compareable in the way we'd want for a fair penalty
Therefore, with Ridge/Lasso, we generally want features to be on comparable scales.
```

In [10]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge

model = make_pipeline(
    StandardScaler(),
    Ridge(alpha=1.0)
)

model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('ridge', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](3,)","['x1','x2','x3']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,3
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


**Note:** 
```
It's true that - Ordinary Linear Regression generally doesn't require scaling.
But Regularized models like Ridge and Lasso are generally much more sensitive to feature scale. and that's why scalling becomes important.
```

***Ridge is particularly useful when you have correlated features and want to stabilize the model.***

## Ridge vs Linear Regression
|                                | Linear Regression | Ridge                 |
| ------------------------------ | ----------------- | --------------------- |
| Objective                      | Minimize error    | Error + L2 penalty    |
| Regularization                 | ❌               | ✅                     |
| Large coefficients             | Can occur         | Penalized             |
| Coefficients                   | Unconstrained     | Shrunk                |
| Coefficients become exactly 0? | —                 | Generally no          |
| Overfitting control            | Limited           | Better                |
| Scaling required?              | Usually no        | Generally recommended |
| Main hyperparameter            | —                 | `alpha`               |


***Ridge vs Polynomial Regression***
- Polynomial adds flexibility but Ridge controls complexity.
- It's combination can be powerful.

## Exercise

In [12]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import numpy as np

for alpha in [0.001, 0.01, 0.1, 1, 10, 100]:

    model = make_pipeline(
        PolynomialFeatures(degree=10),
        StandardScaler(),
        Ridge(alpha=alpha)
    )

    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_rmse = np.sqrt(
        mean_squared_error(y_train, train_pred)
    )

    test_rmse = np.sqrt(
        mean_squared_error(y_test, test_pred)
    )

    print(
        f"Alpha: {alpha}, "
        f"Train RMSE: {train_rmse:.2f}, "
        f"Test RMSE: {test_rmse:.2f}"
    )

Alpha: 0.001, Train RMSE: 0.51, Test RMSE: 8.97
Alpha: 0.01, Train RMSE: 0.59, Test RMSE: 3.09
Alpha: 0.1, Train RMSE: 0.66, Test RMSE: 1.04
Alpha: 1, Train RMSE: 0.76, Test RMSE: 0.92
Alpha: 10, Train RMSE: 0.98, Test RMSE: 1.18
Alpha: 100, Train RMSE: 1.93, Test RMSE: 2.44


## Final Outcome -
> Ridge is Linear Regression with L2 regularization. It penalizes large coefficients, which reduces model complexity and can improve generalization, especially when features are numerous or correlated. Unlike Lasso, Ridge generally keeps all features rather than setting some coefficients exactly to zero.